In [1]:
import polars as pl
from pybiomart import Server
import os
import numpy as np
import pandas as pd

In [2]:
WORKING_DIR = '/group/pmc021/amunif/epi-thesis/workflow/08_HepG2'
DATASET_DIR = '/group/pmc021/amunif/epi-thesis/dataset'

In [3]:
def query_ensembl_by_location(item):

    chrom = item[17]
    chromosome = item[14]
    start = item[15]
    end = item[16]
    test_id = item[0]
    gene_id = item[1]
    locus = item[3]
    
    # Connect to the GRCh37 archive server
    server = Server(host='http://grch37.ensembl.org')
    
    # Select the human dataset
    dataset = (server.marts['ENSEMBL_MART_ENSEMBL']
               .datasets['hsapiens_gene_ensembl'])

    result = dataset.query(attributes=['chromosome_name', 
                                   'start_position', 
                                   'end_position', 
                                   'strand',
                                   'external_gene_name', 
                                   'ensembl_gene_id',
                                   'gene_biotype'],
                       filters={'chromosome_name': chrom,
                                'start': start,
                                'end': end})
    
    result['tss'] = result.apply(lambda row: row['Gene start (bp)'] if row['Strand'] == 1 else row['Gene end (bp)'], axis=1)
    result['tss -2kb'] = result['tss'] - 2000
    result['tss +2kb'] = result['tss'] + 2000
    result['test_id'] = test_id
    result['gene_id'] = gene_id
    result['locus'] = locus
    result['chromosome'] = chromosome
    result['orig_start'] = start
    result['orig_end'] = end
    return result

In [4]:
# Open file
df = pl.read_csv(os.path.join(DATASET_DIR, 'GSM3718064_HepG2_exp.txt'), separator="\t")

In [5]:
# Split locus column into 3 columns: chr, start, end
df = df.with_columns([
    pl.col("locus").str.split(":").list.get(0).alias("chromosome"),
    pl.col("locus").str.split(":").list.get(1).str.split_exact("-", 1).struct.rename_fields(["start", "end"]).alias("position")
])

# Unnest the column and convert into integer
df = df.unnest("position")

df = df.with_columns([
    pl.col("start").cast(pl.Int64),
    pl.col("end").cast(pl.Int64)
])

df = df.with_columns([
    pl.col('chromosome').str.replace('chr', '').alias('chr')
])

df

test_id,gene_id,gene,locus,sample_1,sample_2,status,value_1,value_2,log2(fold_change),test_stat,p_value,q_value,significant,chromosome,start,end,chr
str,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,str,str,i64,i64,str
"""XLOC_000001""","""XLOC_000001""","""OR4F5""","""chr1:69090-70008""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no""","""chr1""",69090,70008,"""1"""
"""XLOC_000002""","""XLOC_000002""","""LOC100132062,LOC100133331""","""chr1:323891-328581""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0473925,0.0341591,-0.472389,0.0,1.0,1.0,"""no""","""chr1""",323891,328581,"""1"""
"""XLOC_000003""","""XLOC_000003""","""OR4F29""","""chr1:367658-368597""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no""","""chr1""",367658,368597,"""1"""
"""XLOC_000004""","""XLOC_000004""","""LOC643837""","""chr1:761585-794889""","""hepg2_hr2""","""hepg2_hr3""","""OK""",2.46857,2.8058,0.184736,0.455074,0.63585,0.999565,"""no""","""chr1""",761585,794889,"""1"""
"""XLOC_000005""","""XLOC_000005""","""-""","""chr1:840263-843900""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no""","""chr1""",840263,843900,"""1"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""XLOC_030028""","""XLOC_030028""","""-""","""chrY:13319431-13324829""","""hepg2_hr2""","""hepg2_hr3""","""OK""",0.337879,0.457897,0.438516,1.13793,0.25685,0.999565,"""no""","""chrY""",13319431,13324829,"""Y"""
"""XLOC_030029""","""XLOC_030029""","""-""","""chrY:13325034-13326120""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.396888,0.357658,-0.150148,0.0,1.0,1.0,"""no""","""chrY""",13325034,13326120,"""Y"""
"""XLOC_030030""","""XLOC_030030""","""-""","""chrY:13331089-13332358""","""hepg2_hr2""","""hepg2_hr3""","""OK""",0.477227,0.127331,-1.90609,-1.86192,0.1249,0.999565,"""no""","""chrY""",13331089,13332358,"""Y"""


In [6]:
df_np = df.to_numpy()

In [7]:
df_np

array([['XLOC_000001', 'XLOC_000001', 'OR4F5', ..., 69090, 70008, '1'],
       ['XLOC_000002', 'XLOC_000002', 'LOC100132062,LOC100133331', ...,
        323891, 328581, '1'],
       ['XLOC_000003', 'XLOC_000003', 'OR4F29', ..., 367658, 368597, '1'],
       ...,
       ['XLOC_030030', 'XLOC_030030', '-', ..., 13331089, 13332358, 'Y'],
       ['XLOC_030031', 'XLOC_030031', '-', ..., 15053270, 15053343, 'Y'],
       ['XLOC_030032', 'XLOC_030032', '-', ..., 59001259, 59003893, 'Y']],
      dtype=object)

In [8]:
df_np.shape

(30032, 18)

In [9]:
df_np[0]

array(['XLOC_000001', 'XLOC_000001', 'OR4F5', 'chr1:69090-70008',
       'hepg2_hr2', 'hepg2_hr3', 'NOTEST', 0.0, 0.0, 0.0, 0.0, 1.0, 1.0,
       'no', 'chr1', 69090, 70008, '1'], dtype=object)

In [10]:
output_path = os.path.join(WORKING_DIR, 'dataset', 'ensembl_by_test_id.csv')

i = 1
total = df_np.shape[0]

for item in df_np:

    if ((i % 100 == 0) or (i == total)):
        print(f"{i}/{total} - {item[0]}")
    
    result = query_ensembl_by_location(item)
    result.to_csv(output_path, mode='a', header=(not os.path.exists(output_path)), index=False)
    i += 1

100/30032 - XLOC_000100
200/30032 - XLOC_000200
300/30032 - XLOC_000300
400/30032 - XLOC_000400
500/30032 - XLOC_000500
600/30032 - XLOC_000600
700/30032 - XLOC_000700
800/30032 - XLOC_000800
900/30032 - XLOC_000900
1000/30032 - XLOC_001000
1100/30032 - XLOC_001100
1200/30032 - XLOC_001200
1300/30032 - XLOC_001300
1400/30032 - XLOC_001400
1500/30032 - XLOC_001500
1600/30032 - XLOC_001600
1700/30032 - XLOC_001700
1800/30032 - XLOC_001800
1900/30032 - XLOC_001900
2000/30032 - XLOC_002000
2100/30032 - XLOC_002100
2200/30032 - XLOC_002200
2300/30032 - XLOC_002300
2400/30032 - XLOC_002400
2500/30032 - XLOC_002500
2600/30032 - XLOC_002600
2700/30032 - XLOC_002700
2800/30032 - XLOC_002800
2900/30032 - XLOC_002900
3000/30032 - XLOC_003000
3100/30032 - XLOC_003100
3200/30032 - XLOC_003200
3300/30032 - XLOC_003300
3400/30032 - XLOC_003400
3500/30032 - XLOC_003500
3600/30032 - XLOC_003600
3700/30032 - XLOC_003700
3800/30032 - XLOC_003800
3900/30032 - XLOC_003900
4000/30032 - XLOC_004000
4100/3003

In [11]:
schema = pl.Schema({
    "Chromosome/scaffold name"  : pl.String(),
    "Gene start (bp)"           : pl.Int64(),
    "Gene end (bp)"             : pl.Int64(),
    "Strand"                    : pl.Int64(),
    "Gene name"                 : pl.String(),
    "Gene stable ID"            : pl.String(),
    "Gene type"                 : pl.String(),
    "tss"                       : pl.Int64(),
    "tss -2kb"                  : pl.Int64(),
    "tss +2kb"                  : pl.Int64(),
    "test_id"                   : pl.String(),
    "gene_id"                   : pl.String(),
    "locus"                     : pl.String(),
    "chromosome"                : pl.String(),
    "orig_start"                : pl.Int64(),
    "orig_end"                  : pl.Int64(),
})

In [12]:
ensembl_df = pl.read_csv(os.path.join(WORKING_DIR, "dataset", "ensembl_by_gen_id.csv"), schema=schema)

In [13]:
ensembl_df

Chromosome/scaffold name,Gene start (bp),Gene end (bp),Strand,Gene name,Gene stable ID,Gene type,tss,tss -2kb,tss +2kb,test_id,gene_id,locus,chromosome,orig_start,orig_end
str,i64,i64,i64,str,str,str,i64,i64,i64,str,str,str,str,i64,i64
"""1""",69091,70008,1,"""OR4F5""","""ENSG00000186092""","""protein_coding""",69091,67091,71091,"""XLOC_000001""","""XLOC_000001""","""chr1:69090-70008""","""chr1""",69090,70008
"""1""",317720,453948,1,"""RP4-669L17.10""","""ENSG00000237094""","""lincRNA""",317720,315720,319720,"""XLOC_000002""","""XLOC_000002""","""chr1:323891-328581""","""chr1""",323891,328581
"""1""",326096,328112,1,"""RP4-669L17.8""","""ENSG00000250575""","""pseudogene""",326096,324096,328096,"""XLOC_000002""","""XLOC_000002""","""chr1:323891-328581""","""chr1""",323891,328581
"""1""",317720,453948,1,"""RP4-669L17.10""","""ENSG00000237094""","""lincRNA""",317720,315720,319720,"""XLOC_000003""","""XLOC_000003""","""chr1:367658-368597""","""chr1""",367658,368597
"""1""",367640,368634,1,"""OR4F29""","""ENSG00000235249""","""protein_coding""",367640,365640,369640,"""XLOC_000003""","""XLOC_000003""","""chr1:367658-368597""","""chr1""",367658,368597
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Y""",27209230,27246039,-1,"""TTTY4C""","""ENSG00000228296""","""lincRNA""",27246039,27244039,27248039,"""XLOC_030019""","""XLOC_030019""","""chrY:27209229-27246039""","""chrY""",27209229,27246039
"""Y""",27329790,27330920,-1,"""TTTY17C""","""ENSG00000223641""","""lincRNA""",27330920,27328920,27332920,"""XLOC_030020""","""XLOC_030020""","""chrY:27329789-27330920""","""chrY""",27329789,27330920
"""Y""",10037764,10037915,1,"""RNA5-8SP6""","""ENSG00000251705""","""rRNA""",10037764,10035764,10039764,"""XLOC_030025""","""XLOC_030025""","""chrY:10037791-10037910""","""chrY""",10037791,10037910


In [14]:
ensembl_df.select(pl.col('gene_id').n_unique())

gene_id
u32
25929


In [15]:
# Filter only the 'protein_coding' gene
pc_df = ensembl_df.filter(pl.col('Gene type') == 'protein_coding')

In [16]:
pc_df

Chromosome/scaffold name,Gene start (bp),Gene end (bp),Strand,Gene name,Gene stable ID,Gene type,tss,tss -2kb,tss +2kb,test_id,gene_id,locus,chromosome,orig_start,orig_end
str,i64,i64,i64,str,str,str,i64,i64,i64,str,str,str,str,i64,i64
"""1""",69091,70008,1,"""OR4F5""","""ENSG00000186092""","""protein_coding""",69091,67091,71091,"""XLOC_000001""","""XLOC_000001""","""chr1:69090-70008""","""chr1""",69090,70008
"""1""",367640,368634,1,"""OR4F29""","""ENSG00000235249""","""protein_coding""",367640,365640,369640,"""XLOC_000003""","""XLOC_000003""","""chr1:367658-368597""","""chr1""",367658,368597
"""1""",860260,879955,1,"""SAMD11""","""ENSG00000187634""","""protein_coding""",860260,858260,862260,"""XLOC_000006""","""XLOC_000006""","""chr1:851135-917473""","""chr1""",851135,917473
"""1""",861264,866445,-1,"""AL645608.1""","""ENSG00000268179""","""protein_coding""",866445,864445,868445,"""XLOC_000006""","""XLOC_000006""","""chr1:851135-917473""","""chr1""",851135,917473
"""1""",879584,894689,-1,"""NOC2L""","""ENSG00000188976""","""protein_coding""",894689,892689,896689,"""XLOC_000006""","""XLOC_000006""","""chr1:851135-917473""","""chr1""",851135,917473
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Y""",24314689,24329129,-1,"""RBMY1F""","""ENSG00000169800""","""protein_coding""",24329129,24327129,24331129,"""XLOC_030009""","""XLOC_030009""","""chrY:24314688-24329089""","""chrY""",24314688,24329089
"""Y""",25275502,25345241,-1,"""DAZ1""","""ENSG00000188120""","""protein_coding""",25345241,25343241,25347241,"""XLOC_030012""","""XLOC_030012""","""chrY:25275501-25345239""","""chrY""",25275501,25345239
"""Y""",26191376,26194166,-1,"""CDY1B""","""ENSG00000172352""","""protein_coding""",26194166,26192166,26196166,"""XLOC_030014""","""XLOC_030014""","""chrY:26191376-26194161""","""chrY""",26191376,26194161


In [17]:
# Select top 1
pc_top1_df = pc_df.group_by("gene_id", maintain_order=True).agg(pl.all().first())

In [18]:
pc_top1_df.shape

(22154, 16)

In [19]:
pc_top1_df

gene_id,Chromosome/scaffold name,Gene start (bp),Gene end (bp),Strand,Gene name,Gene stable ID,Gene type,tss,tss -2kb,tss +2kb,test_id,locus,chromosome,orig_start,orig_end
str,str,i64,i64,i64,str,str,str,i64,i64,i64,str,str,str,i64,i64
"""XLOC_000001""","""1""",69091,70008,1,"""OR4F5""","""ENSG00000186092""","""protein_coding""",69091,67091,71091,"""XLOC_000001""","""chr1:69090-70008""","""chr1""",69090,70008
"""XLOC_000003""","""1""",367640,368634,1,"""OR4F29""","""ENSG00000235249""","""protein_coding""",367640,365640,369640,"""XLOC_000003""","""chr1:367658-368597""","""chr1""",367658,368597
"""XLOC_000006""","""1""",860260,879955,1,"""SAMD11""","""ENSG00000187634""","""protein_coding""",860260,858260,862260,"""XLOC_000006""","""chr1:851135-917473""","""chr1""",851135,917473
"""XLOC_000007""","""1""",860260,879955,1,"""SAMD11""","""ENSG00000187634""","""protein_coding""",860260,858260,862260,"""XLOC_000007""","""chr1:851135-917473""","""chr1""",851135,917473
"""XLOC_000008""","""1""",948803,949920,1,"""ISG15""","""ENSG00000187608""","""protein_coding""",948803,946803,950803,"""XLOC_000008""","""chr1:948846-949919""","""chr1""",948846,949919
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""XLOC_030009""","""Y""",24314689,24329129,-1,"""RBMY1F""","""ENSG00000169800""","""protein_coding""",24329129,24327129,24331129,"""XLOC_030009""","""chrY:24314688-24329089""","""chrY""",24314688,24329089
"""XLOC_030012""","""Y""",25275502,25345241,-1,"""DAZ1""","""ENSG00000188120""","""protein_coding""",25345241,25343241,25347241,"""XLOC_030012""","""chrY:25275501-25345239""","""chrY""",25275501,25345239
"""XLOC_030014""","""Y""",26191376,26194166,-1,"""CDY1B""","""ENSG00000172352""","""protein_coding""",26194166,26192166,26196166,"""XLOC_030014""","""chrY:26191376-26194161""","""chrY""",26191376,26194161


In [20]:
pc_top1_df.columns

['gene_id',
 'Chromosome/scaffold name',
 'Gene start (bp)',
 'Gene end (bp)',
 'Strand',
 'Gene name',
 'Gene stable ID',
 'Gene type',
 'tss',
 'tss -2kb',
 'tss +2kb',
 'test_id',
 'locus',
 'chromosome',
 'orig_start',
 'orig_end']

In [26]:
# Reorder the column
ordered_cols = ['chromosome', 'tss -2kb', 'tss +2kb', 'test_id', 'gene_id',
                'Chromosome/scaffold name', 'Gene start (bp)', 'Gene end (bp)', 
                'Strand', 'Gene name', 'Gene stable ID', 'Gene type', 'tss', 
                'locus', 'orig_start', 'orig_end']

In [22]:
pc_top1_df = pc_top1_df.select(ordered_cols)

In [23]:
pc_top1_df

chromosome,tss -2kb,tss +2kb,gene_id,test_id,Chromosome/scaffold name,Gene start (bp),Gene end (bp),Strand,Gene name,Gene stable ID,Gene type,tss,locus,orig_start,orig_end
str,i64,i64,str,str,str,i64,i64,i64,str,str,str,i64,str,i64,i64
"""chr1""",67091,71091,"""XLOC_000001""","""XLOC_000001""","""1""",69091,70008,1,"""OR4F5""","""ENSG00000186092""","""protein_coding""",69091,"""chr1:69090-70008""",69090,70008
"""chr1""",365640,369640,"""XLOC_000003""","""XLOC_000003""","""1""",367640,368634,1,"""OR4F29""","""ENSG00000235249""","""protein_coding""",367640,"""chr1:367658-368597""",367658,368597
"""chr1""",858260,862260,"""XLOC_000006""","""XLOC_000006""","""1""",860260,879955,1,"""SAMD11""","""ENSG00000187634""","""protein_coding""",860260,"""chr1:851135-917473""",851135,917473
"""chr1""",858260,862260,"""XLOC_000007""","""XLOC_000007""","""1""",860260,879955,1,"""SAMD11""","""ENSG00000187634""","""protein_coding""",860260,"""chr1:851135-917473""",851135,917473
"""chr1""",946803,950803,"""XLOC_000008""","""XLOC_000008""","""1""",948803,949920,1,"""ISG15""","""ENSG00000187608""","""protein_coding""",948803,"""chr1:948846-949919""",948846,949919
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrY""",24327129,24331129,"""XLOC_030009""","""XLOC_030009""","""Y""",24314689,24329129,-1,"""RBMY1F""","""ENSG00000169800""","""protein_coding""",24329129,"""chrY:24314688-24329089""",24314688,24329089
"""chrY""",25343241,25347241,"""XLOC_030012""","""XLOC_030012""","""Y""",25275502,25345241,-1,"""DAZ1""","""ENSG00000188120""","""protein_coding""",25345241,"""chrY:25275501-25345239""",25275501,25345239
"""chrY""",26192166,26196166,"""XLOC_030014""","""XLOC_030014""","""Y""",26191376,26194166,-1,"""CDY1B""","""ENSG00000172352""","""protein_coding""",26194166,"""chrY:26191376-26194161""",26191376,26194161


In [24]:
pc_top1_df.write_csv(os.path.join(WORKING_DIR, "dataset", "ensembl_top1.csv"), include_header=False, separator="\t")

In [27]:
pc_top1_df.head(100).write_csv(os.path.join(WORKING_DIR, "dataset", "ensembl_100.csv"), include_header=False, separator="\t")